# Suite TSP — The TypeScript Patterns dialect

ECMA-262's binding and assignment patterns (14.3.3, "Destructuring Binding Patterns"; 13.15.5, "Destructuring
Assignment"), the sublanguage a function's, an arrow's, a signature's and a function type's parameters and an
assignment's target hold: the dialect `Expressions.TypeScript.Patterns` of the TypeScript family, whose `Term` was
TypeScript5's `Pattern` union, its branches unchanged, its kinds mbse-programs' patterns, its holder, what a pattern
binds through what holds it, its editions, its printing, and its evaluation, which refuses each kind by name (mbse-basis [CAP-277: TypeScript's types and destructuring patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-277.md);
[CAP-228: the TypeScript5 expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-228.md)).

In [ ]:
import functools

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, TypeScript5, Types as TypeScriptTypes
from mbse.Expressions.TypeScript.Patterns import Expressions as TSP, Text as TSPText
from mbse.Expressions.TypeScript.TypeScript5 import Simplifiers as TV, Expressions as TS, Text as TSText
from mbse.Expressions.TypeScript.Types import Expressions as TST
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = Builders.Store()  # the framework, the standard dialects TypeScript5's domains use, and TypeScript's family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
                TypeScriptTypes.install, TypeScriptPatterns.install]:
    store.transaction(install)
store.transaction(TypeScriptPatterns.define)  # the versions of TypeScript 5 the Patterns dialect covers


READER = {d.union: d for d in (TS.DIALECT, TST.DIALECT, TSP.DIALECT)}  # the dialect whose union holds a term


def held(spec, dialect=TSP.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
    transaction."""
    with store.transaction():
        return functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression(
            lambda e: e.root(dialect.spec(spec)))


def pattern(spec):
    """A term of the Patterns dialect, held by its holder."""
    return F.term(held(spec))


def ts(spec):
    """A term of TypeScript5, held by its holder."""
    return F.term(held(spec, TS.DIALECT))


def name(text):
    """A spec of the identifier `text`, in any union that has identifiers."""
    return lambda e: e.identifier(lambda i: i.name(text))


one = lambda e: e.literal(lambda li: li.value(lambda v: v.as_scalar(1.0)))  # noqa: E731
# [a, , ...r]: a pattern of every kind but a parameter property, and the terms a target may be
every = lambda p: p.objectpattern(lambda o: o.properties([  # noqa: E731: { k: [a, , ...r] = c, m: x.y, ...z }
    lambda q: q.property(lambda s: s.propertyKind("init").key(name("k")).value(lambda v: v.assignmentpattern(lambda a: a.left(
        lambda t: t.arraypattern(lambda r: r.elements([name("a"), lambda e: e.elision(lambda h: h), lambda e: e.restelement(
            lambda s: s.argument(name("r")))]))).right(name("c"))))),
    lambda q: q.property(lambda s: s.propertyKind("init").key(name("m")).value(lambda v: v.memberexpression(
        lambda m: m.object(name("x")).property(name("y"))))),
    lambda q: q.restelement(lambda r: r.argument(name("z")))]))

<a id="TSP-01"></a>
## TSP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the properties that hold it as a subterm, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
# Its Term: its four patterns, the terms of TypeScript5 a target may be, a parameter property and invalid syntax, as
# TypeScript5's `Pattern` union had them; each own kind's schema is named by its path; it has no helper of its own.
union = [b.name for b in store.registered(TSP.DIALECT.union).branches]
assert union == [*TSP.TAGS[:4], *TSP.TERMS, "tsparameterproperty", "error", "missing"] and TSP.HELPERS == ()
assert TSP.TAGS == ("arraypattern", "objectpattern", "assignmentpattern", "restelement", "tsparameterproperty")
assert TSP.DIALECT.name() == "TypeScript Patterns" and TSP.DIALECT.tag == "typescript_patterns"
assert TSP.DIALECT.path == "TypeScript.Patterns" and TSP.DIALECT.union == "Expressions.TypeScript.Patterns.Term"
Patterns = store.registered(TSP.DIALECT.union)
kinds = {b.name: b.schema for b in Patterns.branches}
assert [kinds[tag].name for tag in TSP.TAGS] == [f"Expressions.TypeScript.Patterns.{kind}" for kind in (
    "ArrayPattern", "ObjectPattern", "AssignmentPattern", "RestElement", "TSParameterProperty")]
assert all(kinds[tag].name.startswith("Expressions.TypeScript.TypeScript5.") for tag in TSP.TERMS)  # TypeScript5's
assert {tag: list(kinds[tag].properties) for tag in TSP.TAGS} == {
    "arraypattern": ["decorators", "elements", "optional", "typeAnnotation"],
    "objectpattern": ["decorators", "properties", "optional", "typeAnnotation"],
    "assignmentpattern": ["decorators", "left", "optional", "typeAnnotation", "right"],
    "restelement": ["decorators", "argument", "optional", "typeAnnotation", "value"],
    "tsparameterproperty": ["decorators", "accessibility", "static", "override", "readonly", "parameter"]}
assert F._parts(kinds["arraypattern"]) == ((), ("decorators", "elements"), ("optional", "typeAnnotation"))
assert F._parts(kinds["assignmentpattern"]) == (("left", "right"), ("decorators",), ("optional", "typeAnnotation"))
assert kinds["assignmentpattern"].properties["right"].schema is store.registered(TS.DIALECT.union)  # a default, an expression
assert kinds["restelement"].properties["value"].schema is kinds["assignmentpattern"]  # a helper, as mbse-programs has it
# None refines Let: a pattern binds through what holds it
assert not any(F.is_binder(kinds[tag]) for tag in TSP.TAGS)
# Where it is held: a function's, an arrow's, a signature's, a function type's and an index signature's parameters, an
# assignment's target; and TypeScript5's Term lists three of its kinds, a property's value holding them
expressions = {b.name: b.schema for b in store.registered(TS.DIALECT.union).branches}
types = {b.name: b.schema for b in store.registered(TST.DIALECT.union).branches}
signatures = {b.name: b.schema for b in store.registered("Expressions.TypeScript.Types.TypeElement").branches}
for schema, prop in [(expressions["functionexpression"], "params"), (expressions["arrowfunctionexpression"], "params"),
                     (expressions["tsemptybodyfunctionexpression"], "params"), (types["tsfunctiontype"], "params"),
                     (types["tsconstructortype"], "params"), (signatures["tsmethodsignature"], "params"),
                     (signatures["tscallsignaturedeclaration"], "params"),
                     (signatures["tsconstructsignaturedeclaration"], "params"), (signatures["tsindexsignature"], "parameters")]:
    assert schema.properties[prop].schema.element is Patterns, (schema.name, prop)
assert expressions["assignmentexpression"].properties["left"].schema is Patterns
assert [expressions[tag] for tag in TSP.TAGS[:3]] == [kinds[tag] for tag in TSP.TAGS[:3]]
# Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
# round-tripped
Holder = store.registered("Expressions.TypeScript.Patterns.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is Patterns and list(Holder.adjacencies) == ["abi"] and Holder.validate() == []
target = held(every)
assert Validators.Validate(store)(Holder, target) == []
with store.transaction():
    linked = store.Schemas.OfExpression(lambda s: s.name("tsp.every").expression(lambda a: a.insert(lambda e: e.expression(target))))
assert [F.same(entry.expression, target) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(decode(Holder, encode(Holder, target)).root, target.root)
    assert F.same(decode(Patterns, encode(Patterns, F.term(target))), F.term(target))
# Its install runs before or after its family's: each names the others' unions and helpers, created where not installed
alone = Builders.Store()
for install in [Framework.install, TypeScriptPatterns.install, TypeScript5.install, TypeScriptTypes.install]:
    alone.transaction(install)
assert [b.name for b in alone.registered(TSP.DIALECT.union).branches] == union

<a id="TSP-02"></a>
## TSP-02 · Each kind's checks; forms and `make`; what a pattern binds, through what holds it: a function's parameters within it, an assignment's target a reference; simplification leaves each kind in the residual, named

In [ ]:
# Each kind's checks: the subterms it needs, the identifiers within it, its annotation's type
for spec, problems in [
    (lambda p: p.assignmentpattern(lambda a: a.left(name("a"))), ["an assignmentpattern needs a right"]),
    (lambda p: p.restelement(lambda r: r), ["a restelement needs an argument"]),
    (lambda p: p.tsparameterproperty(lambda t: t.accessibility("private")), ["a tsparameterproperty needs a parameter"]),
    (lambda p: p.arraypattern(lambda a: a.elements([name("a b")])), ["elements 0: an identifier's name cannot be 'a b'"]),
    (lambda p: p.objectpattern(lambda o: o.typeAnnotation(lambda a: a.typeAnnotation(lambda t: t.tsuniontype(
        lambda u: u.types([lambda k: k.tsanykeyword(lambda h: h)]))))), ["typeAnnotation typeAnnotation: a tsuniontype needs two types"]),  # its annotation's type, by its path
    (lambda p: p.arraypattern(lambda a: a.elements([lambda e: e.error(lambda r: r.text("1 +"))])), ["elements 0: invalid syntax '1 +'"]),  # its containers' terms, by container
]:
    assert TSP.DIALECT.validate(pattern(spec), {"a"}) == problems, (TSP.DIALECT.validate(pattern(spec), {"a"}), problems)
# Its kinds read and written by their forms: a pattern's containers, its decorators and its elements, each its size
for node in F.walk(pattern(every)):
    reader = READER[F.dialect_of(node)]  # made by the dialect whose union holds it, in its holder
    assert F.same(F.term(held(reader.make(F.form(node)), reader)), node)
decorated = pattern(lambda p: p.arraypattern(lambda a: a.decorators([lambda d: d.expression(name("d"))]).elements([name("a"), name("b")])))
form = F.form(decorated)
assert form.kind == "arraypattern" and form.attributes == {"decorators": F.Part("decorators", {}, 1),
                                                           "elements": F.Part("elements", {}, 2)}
assert [F.name_of(a) for a in form.arguments] == ["d", "a", "b"] and F.same(F.term(held(TSP.DIALECT.make(form))), decorated)
with raises(ValueError, match="an arraypattern's decorators, elements hold 3 arguments, got 2"):
    held(TSP.DIALECT.make(F.Form("arraypattern", form.attributes, form.arguments[:2])))
# What a pattern binds: the names of its identifiers, within arrays, objects, defaults, rests and parameter properties;
# a member, or any other target, binds none
assert TSP.bound_names(pattern(every)) == {"a", "r", "z"} and TSP.bound_names is TS.bound_names
assert TSP.bound_names(pattern(lambda p: p.tsparameterproperty(lambda t: t.parameter(lambda a: a.assignmentpattern(
    lambda s: s.left(name("q")).right(one)))))) == {"q"}
assert TSP.bound_names(pattern(lambda p: p.identifier(lambda i: i))) == frozenset()  # a name not written
assert TSP.bound_names(pattern(lambda p: p.memberexpression(lambda m: m.object(name("x")).property(name("y"))))) == frozenset()
# A pattern binds through what holds it: a function's parameters within it, its defaults among it; an assignment's
# target is a reference to what it assigns
arrow = ts(lambda u: u.arrowfunctionexpression(lambda a: a.params([every, lambda p: p.assignmentpattern(lambda s: s.left(name("b")).right(name("a")))]).body(
    lambda b: b.binaryexpression(lambda e: e.left(name("a")).operator("+").right(name("b"))))))
assert SY.free(arrow, TS.DIALECT) == {"c", "x"} and TS.DIALECT.validate(arrow, {"c", "x"}) == []  # a default, a member's object
assign = ts(lambda u: u.assignmentexpression(lambda a: a.left(every).operator("=").right(name("o"))))
assert SY.free(assign, TS.DIALECT) == {"a", "r", "c", "x", "z", "o"}
assert TS.DIALECT.validate(assign, {"a", "r", "c", "x", "o"}) == ["left: properties 2: argument: identifier 'z' is not bound"]
assert SY.free(pattern(every), TSP.DIALECT) == {"a", "r", "c", "x", "z"}  # a pattern alone, held by its holder: references
# Evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does; inference knows nothing of it
for spec in [every, lambda p: p.restelement(lambda r: r.argument(name("a")))]:
    with raises(NotEvaluated, match=f"TypeScript Patterns {F.kind_of(pattern(spec))[0]} is not simplified"):
        TV.OfAny(pattern(spec)).evaluate()
with raises(NotEvaluated, match="the root is not a literal: a is not bound"):  # a name, TypeScript5's kind, by its scope
    TV.OfAny(pattern(name("a"))).evaluate()
assert TSP.DIALECT.infer(pattern(lambda p: p.restelement(lambda r: r.argument(name("a")))), {"a": D.Anything}).name() == "any"

<a id="TSP-03"></a>
## TSP-03 · Its editions: the versions of TypeScript 5, 5.0 to 5.9, by their published names, each with every one of its own kinds

In [ ]:
tsp = store.Instances.Expressions.TypeScript.Patterns
VERSIONS = tuple(f"TypeScript 5.{minor}" for minor in range(10))
assert tuple(FS.standards(tsp)) == VERSIONS and TSP.SINCE == {}
assert FS.properties(FS.find(tsp, "TypeScript 5.9")) == {"bodies": ("com.microsoft",), "revision": ("version", "5.9")}
assert [s.name for s in FS.kinds(FS.find(tsp, "TypeScript 5.0"))] == [kinds[tag].name for tag in TSP.TAGS]  # its own

<a id="TSP-04"></a>
## TSP-04 · Printed as TypeScript writes patterns, from its own holder and within TypeScript5's, by the family's printer

In [ ]:
assert TSPText.ToText is TSText.ToText and set(TSPText.TEXTS) == set(TSP.TAGS)
for spec, text in [
    (every, "({ k: [a,, ...r] = c, m: x.y, ...z })"),  # as a statement holds it
    (lambda p: p.arraypattern(lambda a: a.decorators([lambda d: d.expression(name("d"))]).elements([name("a"),
        lambda e: e.elision(lambda h: h)]).optional(True).typeAnnotation(lambda t: t.typeAnnotation(lambda k: k.tsarraytype(
            lambda r: r.elementType(lambda n: n.tsnumberkeyword(lambda h: h)))))), "@d [a,,]?: number[]"),
    (lambda p: p.tsparameterproperty(lambda t: t.accessibility("public").readonly(True).parameter(lambda a: a.assignmentpattern(
        lambda s: s.left(name("q")).right(one)))), "public readonly q = 1"),
    (lambda p: p.restelement(lambda r: r.argument(name("a")).typeAnnotation(lambda t: t.typeAnnotation(lambda k: k.tsanykeyword(lambda h: h)))),
     "...a: any"),
    (lambda p: p.missing(lambda m: m.kind("pattern")), ""),
]:
    assert TSText.ToText(pattern(spec)) == text, (TSText.ToText(pattern(spec)), text)
with raises(ValueError, match="TypeScript Patterns error has no text to print"):  # refused by its dialect's name
    TSText.ToText(pattern(lambda p: p.error(lambda e: e)))
assert TSText.ToText(ts(lambda u: u.assignmentexpression(lambda a: a.left(every).operator("=").right(name("o"))))) == (
    "({ k: [a,, ...r] = c, m: x.y, ...z } = o)")  # an assignment's target, as a statement holds it
assert TSText.ToText(ts(lambda u: u.functionexpression(lambda f: f.params([lambda p: p.tsparameterproperty(lambda t: t.accessibility(
    "private").parameter(name("a"))), lambda p: p.restelement(lambda r: r.argument(name("b")))])))) == "(function(private a, ...b) {})"